In [1]:
# import sys
# print(sys.version)
# !pip install ipykernel
# !python -m ipykernel install --user --name python310 --display-name "Python 3.11"
# !pip uninstall sentence-transformers torch torchvision -y

# !pip install pandas numpy scikit-learn sentence-transformers torch
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [2]:
# import sys
# print(sys.executable)

In [3]:
# import subprocess
# subprocess.run([r"c:\Users\josha\AppData\Local\Programs\Python\Python311\python.exe", 
#                 "-m", "pip", "install", "torch", "torchvision", "torchaudio", 
#                 "--index-url", "https://download.pytorch.org/whl/cu121"])

In [4]:
# import sys
# !{sys.executable} -m pip install pandas numpy scikit-learn sentence-transformers

In [5]:
# import sys
# !{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [6]:
# import sys
# !{sys.executable} -m pip install datasets

In [7]:
# import sys
# !{sys.executable} -m pip install "accelerate>=1.1.0"

In [8]:
import torch
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Видеокарта: {torch.cuda.get_device_name(0)}")
print(f"Версия CUDA: {torch.version.cuda}")

CUDA доступна: True
Видеокарта: NVIDIA GeForce RTX 4070
Версия CUDA: 12.1


In [9]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CECorrelationEvaluator
from torch.utils.data import DataLoader
from sentence_transformers import InputExample
import math
import warnings
warnings.filterwarnings('ignore')

c:\Users\josha\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
vacancies = pd.read_csv("vacancies2.csv")
resumes = pd.read_csv("resumes.csv")
pairs = pd.read_csv("labeled_pairs.csv")

In [11]:
dataset = pairs.merge(
    vacancies[['vacancy_id', 'vacancy_text', 'required_skills', 'profession', 'level']],
    on='vacancy_id',
    how='left'
).merge(
    resumes[['resume_id', 'resume_text', 'hard_skills', 'soft_skills', 'level']],
    on='resume_id',
    how='left'
)
dataset = dataset.rename(columns={
    'level_x': 'vacancy_level',
    'level_y': 'resume_level'
})
dataset['vacancy_level'] = dataset['vacancy_level'].fillna('junior')
dataset['resume_level'] = dataset['resume_level'].fillna('junior')

print(f"{dataset.shape}")
print(f"{dataset.columns.tolist()}")

(360000, 16)
['vacancy_id', 'resume_id', 'vacancy_profession', 'resume_profession', 'similarity', 'label', 'vacancy_salary', 'resume_salary', 'vacancy_text', 'required_skills', 'profession', 'vacancy_level', 'resume_text', 'hard_skills', 'soft_skills', 'resume_level']


In [12]:
def parse_skills(skills_str):
    if pd.isna(skills_str):
        return []
    try:
        if isinstance(skills_str, str):
            skills_str = skills_str.replace("'", '"')
            return ast.literal_eval(skills_str)
        return skills_str
    except:
        if isinstance(skills_str, str):
            return [s.strip().lower() for s in skills_str.split(',')]
        return []

# Применяем парсинг к колонкам с навыками
dataset['required_skills_list'] = dataset['required_skills'].apply(parse_skills)
dataset['hard_skills_list'] = dataset['hard_skills'].apply(parse_skills)
print(f"Пример: {dataset['required_skills_list'].iloc[0][:3]}")

Пример: ['SEO', 'Копирайтинг', 'Контент']


In [13]:
def balance_dataset(df, samples_per_class=15000):
    balanced_parts = []
    
    for label in [0, 1, 2, 3]:
        label_df = df[df['label'] == label]
        
        if len(label_df) > samples_per_class:
            label_df = label_df.sample(samples_per_class, random_state=42)
        
        balanced_parts.append(label_df)
    
    balanced_df = pd.concat(balanced_parts, ignore_index=True)
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True) 
    return balanced_df

dataset_balanced = balance_dataset(dataset, samples_per_class=15000)
print(dataset_balanced['label'].value_counts().sort_index())

label
0    15000
1    15000
2    15000
3    13444
Name: count, dtype: int64


In [14]:
train_df, temp_df = train_test_split(
    dataset_balanced,
    test_size=0.3,
    stratify=dataset_balanced['label'],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=42
)

print(f"Обучающая выборка: {len(train_df)}")
print(f"Валидационная: {len(valid_df)}")
print(f"Тестовая: {len(test_df)}")

Обучающая выборка: 40910
Валидационная: 8767
Тестовая: 8767


In [15]:

def prepare_samples(df):
    samples = []
    
    for _, row in df.iterrows():
        vac_text = str(row['vacancy_text'])[:1000].replace('\n', ' ').replace('\r', ' ')
        res_text = str(row['resume_text'])[:1000].replace('\n', ' ').replace('\r', ' ')
        
        samples.append(
            InputExample(
                texts=[vac_text, res_text],
                label=float(row['label']) / 3.0
            )
        )   
    return samples
    
train_samples = prepare_samples(train_df)
valid_samples = prepare_samples(valid_df)
test_samples = prepare_samples(test_df)

print(f"Train примеров: {len(train_samples)}")
print(f"Valid примеров: {len(valid_samples)}")
print(f"Test примеров: {len(test_samples)}")

Train примеров: 40910
Valid примеров: 8767
Test примеров: 8767


In [16]:

train_dataloader = DataLoader(
    train_samples, 
    shuffle=True,
    batch_size=32
)

print(f"Количество батчей: {len(train_dataloader)}")

Количество батчей: 1279


In [17]:

model = CrossEncoder(
    "DiTy/cross-encoder-russian-msmarco",
    num_labels=1,
    max_length=512
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8370.12it/s]


In [18]:
evaluator = CECorrelationEvaluator.from_input_examples(
    valid_samples,
    name="validation"
)

In [19]:
warmup_steps = math.ceil(len(train_dataloader) * 3 * 0.1)

print(f"Warmup steps: {warmup_steps}")

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=3,                
    evaluation_steps=500,    
    warmup_steps=warmup_steps,
    output_path="model1",     
    use_amp=True,             
    show_progress_bar=True    
)

print("Обучение завершено!")

Warmup steps: 384


Step,Training Loss,Validation Loss,Validation Pearson,Validation Spearman
500,0.669580,No log,0.550696,0.545837
1000,0.590383,No log,0.638184,0.618615
1500,0.566533,No log,0.664241,0.651408
2000,0.549973,No log,0.692629,0.680140
2500,0.540394,No log,0.708232,0.695479
3000,0.517679,No log,0.722481,0.710435
3500,0.519145,No log,0.731209,0.722508
3837,0.519145,No log,0.739166,0.730105


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.49s/it]


Обучение завершено!


In [ ]:

def predict_and_evaluate(model, test_samples, test_df):
    test_texts1 = [s.texts[0] for s in test_samples]
    test_texts2 = [s.texts[1] for s in test_samples]
    
    predictions = model.predict([(t1, t2) for t1, t2 in zip(test_texts1, test_texts2)])

    pred_classes = (predictions * 3).round().clip(0, 3).astype(int)

    true_classes = [s.label for s in test_samples]
    true_classes_int = [int(round(t * 3)) for t in true_classes]

    print(classification_report(true_classes_int, pred_classes, digits=3))
    
    f1_scores = f1_score(true_classes_int, pred_classes, average=None)
    for i, f1 in enumerate(f1_scores):
        print(f"Class {i}: F1 = {f1:.3f}")
    
    macro_f1 = f1_score(true_classes_int, pred_classes, average='macro')
    weighted_f1 = f1_score(true_classes_int, pred_classes, average='weighted')
    acc = accuracy_score(true_classes_int, pred_classes)
    
    print(f"\nAccuracy: {acc:.3f}")
    print(f"Macro F1: {macro_f1:.3f}")
    print(f"Weighted F1: {weighted_f1:.3f}")
    
    return pred_classes, true_classes_int

# Запускаем оценку
pred_classes, true_classes = predict_and_evaluate(model, test_samples, test_df)


CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0      0.788     0.676     0.728      2250
           1      0.502     0.479     0.490      2250
           2      0.492     0.797     0.608      2250
           3      0.756     0.391     0.516      2017

    accuracy                          0.591      8767
   macro avg      0.634     0.586     0.585      8767
weighted avg      0.631     0.591     0.587      8767


F1 SCORE BY CLASS
Class 0: F1 = 0.728
Class 1: F1 = 0.490
Class 2: F1 = 0.608
Class 3: F1 = 0.516

Accuracy: 0.591
Macro F1: 0.585
Weighted F1: 0.587


In [ ]:


def baseline_predict(row):
    vac_skills = set(row['required_skills_list'])
    res_skills = set(row['hard_skills_list'])
    overlap = len(vac_skills & res_skills)
    
    if overlap >= 3:
        return 3
    elif overlap == 2:
        return 2
    elif overlap == 1:
        return 1
    else:
        return 0

baseline_preds = test_df.apply(baseline_predict, axis=1)
print(classification_report(test_df['label'], baseline_preds, digits=3))


BASELINE: Skill Overlap Only
              precision    recall  f1-score   support

           0      0.300     0.999     0.461      2250
           1      0.350     0.124     0.183      2250
           2      0.012     0.002     0.003      2250
           3      1.000     0.065     0.122      2017

    accuracy                          0.304      8767
   macro avg      0.415     0.297     0.192      8767
weighted avg      0.400     0.304     0.194      8767



In [ ]:


test_df_copy = test_df.copy()
test_df_copy['pred_label'] = pred_classes
test_df_copy['true_label'] = true_classes

errors = test_df_copy[test_df_copy['pred_label'] != test_df_copy['true_label']]

print(f"\nВсего ошибок: {len(errors)} из {len(test_df_copy)} ({len(errors)/len(test_df_copy)*100:.1f}%)")

print("\nПримеры ошибок (модель предсказала НЕВЕРНО):")
for i, (idx, row) in enumerate(errors.head(5).iterrows()):
    print(f"\n--- Ошибка {i+1} ---")
    print(f"Реальный класс: {row['true_label']}, Предсказанный: {row['pred_label']}")
    print(f"Вакансия: {row['vacancy_profession'][:50]}")
    print(f"Резюме: {row['resume_profession'][:50]}")
    vac_skills = set(row['required_skills_list'])
    res_skills = set(row['hard_skills_list'])
    print(f"Пересечение навыков: {len(vac_skills & res_skills)}")


ANALYSIS OF MISCLASSIFICATIONS

Всего ошибок: 3586 из 8767 (40.9%)

Примеры ошибок (модель предсказала НЕВЕРНО):

--- Ошибка 1 ---
Реальный класс: 2, Предсказанный: 1
Вакансия: специалист поддержки компания: ооо онлайн решения 
Резюме: системный администратор
Пересечение навыков: 0

--- Ошибка 2 ---
Реальный класс: 0, Предсказанный: 1
Вакансия: специалист поддержки
Резюме: графический дизайнер
Пересечение навыков: 0

--- Ошибка 3 ---
Реальный класс: 3, Предсказанный: 2
Вакансия: специалист поддержки компания: ооо тех лаб город: 
Резюме: hr-менеджер
Пересечение навыков: 0

--- Ошибка 4 ---
Реальный класс: 2, Предсказанный: 1
Вакансия: системный администратор
Резюме: специалист поддержки
Пересечение навыков: 0

--- Ошибка 5 ---
Реальный класс: 0, Предсказанный: 1
Вакансия: системный администратор
Резюме: маркетолог
Пересечение навыков: 0


In [ ]:
model.save_pretrained("Model1")
print("\nМодель сохранена в папку 'Model1'")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it]


Модель сохранена в папку 'Model1'


In [ ]:

def predict_match(vacancy_text, resume_text, model):
    
    prediction = model.predict([(vacancy_text, resume_text)])[0]

    pred_class = int(round(prediction * 3))
    pred_class = max(0, min(3, pred_class))

    interpretations = {
        0: "Не подходит",
        1: "Слабое соответствие",
        2: "Хорошее соответствие",
        3: "Отличное соответствие"
    }
    
    confidence = abs(prediction - (pred_class / 3))
    confidence_score = 1 - min(confidence * 2, 1)
    
    return pred_class, interpretations[pred_class], confidence_score

print("\nМодель готова к использованию!")



Модель готова к использованию!

Пример использования:

vacancy = "Требуется контент-менеджер со знанием SEO"
resume = "Опыт работы контент-менеджером, владею SEO"
class_num, desc, conf = predict_match(vacancy, resume, model)
print(f"Класс: {class_num}, Описание: {desc}, Уверенность: {conf:.2f}")



МОДЕЛЬКА 2

In [43]:
import pandas as pd
import numpy as np
import ast
import math
import gc
import random
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score
)

from sentence_transformers import (
    CrossEncoder,
    InputExample
)

from torch.utils.data import DataLoader
from transformers import logging


In [44]:

warnings.filterwarnings("ignore")
logging.set_verbosity_error()

random.seed(42)
np.random.seed(42)

In [45]:
vacancies = pd.read_csv("vacancies2.csv")
resumes = pd.read_csv("resumes.csv")
pairs = pd.read_csv("labeled_pairs.csv")

print(f"Vacancies: {len(vacancies)}")
print(f"Resumes: {len(resumes)}")
print(f"Pairs: {len(pairs)}")


Vacancies: 300
Resumes: 1200
Pairs: 360000


In [28]:
vacancies = vacancies[
    [
        'vacancy_id',
        'vacancy_text',
        'required_skills',
        'profession',
        'level'
    ]
]

resumes = resumes[
    [
        'resume_id',
        'resume_text',
        'hard_skills',
        'profession',
        'level'
    ]
]

In [29]:
dataset = pairs.merge(
    vacancies,
    on='vacancy_id',
    how='left'
)

dataset = dataset.merge(
    resumes,
    on='resume_id',
    how='left',
    suffixes=('_vacancy', '_resume')
)
dataset = dataset.loc[
    :,
    ~dataset.columns.duplicated()
]

In [30]:
dataset = dataset.rename(columns={
    'profession_vacancy': 'vacancy_profession',
    'profession_resume': 'resume_profession',
    'level_vacancy': 'vacancy_level',
    'level_resume': 'resume_level'
})

In [31]:
dataset['vacancy_level'] = dataset[
    'vacancy_level'
].fillna('junior')

dataset['resume_level'] = dataset[
    'resume_level'
].fillna('junior')

In [32]:
def parse_skills(skills_str):

    if pd.isna(skills_str):
        return []

    try:
        if isinstance(skills_str, str):
            skills_str = skills_str.replace("'", '"')
            parsed = ast.literal_eval(skills_str)
            if isinstance(parsed, list):
                return [
                    str(x).lower().strip()
                    for x in parsed
                ]
        return []

    except:
        if isinstance(skills_str, str):

            return [
                x.strip().lower()
                for x in skills_str.split(',')
                if x.strip()
            ]
        return []

In [33]:
dataset['required_skills_list'] = dataset[
    'required_skills'
].apply(parse_skills)

dataset['hard_skills_list'] = dataset[
    'hard_skills'
].apply(parse_skills)

In [34]:
dataset['shared_skills'] = dataset.apply(
    lambda row:
    list(
        set(row['required_skills_list'])
        &
        set(row['hard_skills_list'])
    ),
    axis=1
)

dataset['shared_count'] = dataset[
    'shared_skills'
].apply(len)

dataset['skill_coverage'] = dataset.apply(
    lambda row:
    row['shared_count']
    /
    max(len(row['required_skills_list']), 1),
    axis=1
)

In [35]:
dataset['label3'] = dataset['label'].map({
    0: 0,
    1: 0,
    2: 1,
    3: 2
})
print(dataset['label3'].value_counts().sort_index())

label3
0    323136
1     23420
2     13444
Name: count, dtype: int64


In [36]:
MAX_PAIRS_PER_RESUME = 60  
MAX_PAIRS_PER_VACANCY = 60 

balanced_parts = [] 

for label in sorted(dataset['label3'].unique()):
    
    label_df = dataset[dataset['label3'] == label].copy()
    label_df = label_df.sample(frac=1, random_state=42)
    label_df = label_df.groupby('resume_id').head(MAX_PAIRS_PER_RESUME)
    label_df = label_df.groupby('vacancy_id').head(MAX_PAIRS_PER_VACANCY)
    balanced_parts.append(label_df)
    
    print(f"Class {label}: {len(label_df)}")

dataset_balanced = pd.concat(balanced_parts, ignore_index=True)
dataset_balanced = dataset_balanced.loc[:, ~dataset_balanced.columns.duplicated()]


Class 0: 18000
Class 1: 5609
Class 2: 10551


In [37]:
middle_df = dataset_balanced[
    dataset_balanced['label3'] == 1
].copy()

middle_df = middle_df.reset_index(drop=True)

augmented_rows = []

for row in middle_df.itertuples(index=False):

    row_dict = row._asdict()

    augmented_rows.append(dict(row_dict))

    if random.random() < 0.5:

        new_row = dict(row_dict)

        skills = list(new_row['hard_skills_list'])

        if len(skills) > 1:

            random.shuffle(skills)

            new_row['hard_skills_list'] = skills

        augmented_rows.append(new_row)

middle_aug_df = pd.DataFrame(augmented_rows)

middle_aug_df = middle_aug_df.loc[
    :,
    ~middle_aug_df.columns.duplicated()
]

dataset_balanced = pd.concat(
    [
        dataset_balanced,
        middle_aug_df
    ],
    ignore_index=True
)

dataset_balanced = dataset_balanced.reset_index(drop=True)

In [ ]:
class_counts = dataset_balanced[
    'label3'
].value_counts()

print("\nBefore:")
print(class_counts)

target_size = class_counts.max()

equalized_parts = []

for label in sorted(dataset_balanced['label3'].unique()):

    class_df = dataset_balanced[
        dataset_balanced['label3'] == label
    ].copy()

    if len(class_df) < target_size:

        extra = class_df.sample(
            target_size - len(class_df),
            replace=True,
            random_state=42
        )

        class_df = pd.concat(
            [class_df, extra],
            ignore_index=True
        )

    elif len(class_df) > target_size:

        class_df = class_df.sample(
            target_size,
            random_state=42
        )

    equalized_parts.append(class_df)

dataset_balanced = pd.concat(
    equalized_parts,
    ignore_index=True
)

dataset_balanced = dataset_balanced.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nAfter:")
print(
    dataset_balanced['label3']
    .value_counts()
)


Before:
label3
0    18000
1    14119
2    10551
Name: count, dtype: int64

After:
label3
1    18000
2    18000
0    18000
Name: count, dtype: int64


In [39]:
train_df, temp_df = train_test_split(
    dataset_balanced,
    test_size=0.3,
    stratify=dataset_balanced['label3'],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label3'],
    random_state=42
)

print(f"Train: {len(train_df)}")
print(f"Valid: {len(valid_df)}")
print(f"Test: {len(test_df)}")

Train: 37800
Valid: 8100
Test: 8100


In [52]:
def build_texts(row):

    vacancy_skills = ", ".join(
        row.required_skills_list[:8]
    )

    resume_skills = ", ".join(
        row.hard_skills_list[:8]
    )

    shared_skills = ", ".join(
        row.shared_skills[:5]
    )

    vacancy_text = (
        f"Профессия: {row.vacancy_profession}. "
        f"Уровень: {row.vacancy_level}. "
        f"Требуемые навыки: {vacancy_skills}. "
        f"Описание: {str(row.vacancy_text)[:250]}"
    )

    resume_text = (
        f"Профессия: {row.resume_profession}. "
        f"Уровень: {row.resume_level}. "
        f"Навыки: {resume_skills}. "
        f"Общие навыки: {shared_skills}. "
        f"Совпадение навыков: {row.shared_count}. "
        f"Покрытие требований: {row.skill_coverage:.2f}. "
        f"Опыт: {str(row.resume_text)[:250]}"
    )

    return vacancy_text, resume_text


In [55]:
def prepare_samples(df):

    samples = []

    for row in df.itertuples(index=False):

        vacancy_text, resume_text = build_texts(row)

        samples.append(
            InputExample(
                texts=[
                    vacancy_text,
                    resume_text
                ],
                label=int(row.label3)
            )
        )

    return samples

train_samples = prepare_samples(train_df)

print(f"Train samples: {len(train_samples)}")


Train samples: 37800


In [56]:
BATCH_SIZE = 4
EPOCHS = 3

train_dataloader = DataLoader(
    train_samples,
    shuffle=True,
    batch_size=BATCH_SIZE
)
model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    num_labels=3,
    max_length=128,
    automodel_args={
        "ignore_mismatched_sizes": True
    }
)
warmup_steps = math.ceil(
    len(train_dataloader) * EPOCHS * 0.1
)

model.fit(
    train_dataloader=train_dataloader,
    evaluator=None,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path="model2",
    use_amp=True,
    show_progress_bar=True
)

The CrossEncoder `automodel_args` argument was renamed and is now deprecated. Please use `model_kwargs` instead.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10001.86it/s]


Step,Training Loss
500,1.095251
1000,1.017750
1500,0.918927
2000,0.869268
2500,0.835979
3000,0.811942
3500,0.781361
4000,0.724219
4500,0.745925
5000,0.726690


In [57]:
test_pairs = []
true_labels = []

for row in test_df.itertuples(index=False):

    vacancy_text, resume_text = build_texts(row)

    test_pairs.append(
        (
            vacancy_text,
            resume_text
        )
    )

    true_labels.append(row.label3)

predictions = model.predict(
    test_pairs
)

pred_labels = np.argmax(
    predictions,
    axis=1
)


In [59]:
print(
    classification_report(
        true_labels,
        pred_labels,
        digits=3,
        target_names=[
            'Не подходит',
            'Возможен',
            'Подходит'
        ]
    )
)

macro_f1 = f1_score(
    true_labels,
    pred_labels,
    average='macro'
)

weighted_f1 = f1_score(
    true_labels,
    pred_labels,
    average='weighted'
)

accuracy = accuracy_score(
    true_labels,
    pred_labels
)

print(f"\nMacro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")

model.save("model2")

              precision    recall  f1-score   support

 Не подходит      0.882     0.708     0.785      2700
    Возможен      0.705     0.746     0.725      2700
    Подходит      0.774     0.882     0.824      2700

    accuracy                          0.779      8100
   macro avg      0.787     0.779     0.778      8100
weighted avg      0.787     0.779     0.778      8100


Macro F1: 0.7782
Weighted F1: 0.7782
Accuracy: 0.7785


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.30it/s]


Model3

In [13]:
import pandas as pd
import numpy as np
import ast
import math
import gc
import random
import warnings
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CECorrelationEvaluator
from torch.utils.data import DataLoader
from transformers import logging


In [14]:
warnings.filterwarnings("ignore")
logging.set_verbosity_error()
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [15]:
MAX_PAIRS_PER_RESUME = 60
MAX_PAIRS_PER_VACANCY = 60
BATCH_SIZE = 8
MAX_LENGTH = 256
EPOCHS = 3
LEARNING_RATE = 3e-5

In [16]:
MODEL_NAME = "DeepPavlov/rubert-base-cased"

vacancies = pd.read_csv("vacancies2.csv")
resumes = pd.read_csv("resumes.csv")
pairs = pd.read_csv("labeled_pairs.csv")

print(f"Вакансий: {len(vacancies)}")
print(f"Резюме: {len(resumes)}")
print(f"Пар: {len(pairs)}")

Вакансий: 300
Резюме: 1200
Пар: 360000


In [17]:
def parse_skills(skills_str):
    if pd.isna(skills_str):
        return []
    try:
        if isinstance(skills_str, str):
            skills_str = skills_str.replace("'", '"')
            parsed = ast.literal_eval(skills_str)
            if isinstance(parsed, list):
                return [str(x).lower().strip() for x in parsed]
        return []
    except:
        if isinstance(skills_str, str):
            return [x.strip().lower() for x in skills_str.split(',') if x.strip()]
        return []

In [18]:
dataset = pairs.merge(
    vacancies[['vacancy_id', 'vacancy_text', 'required_skills', 'profession', 'level']],
    on='vacancy_id',
    how='left'
)

dataset = dataset.merge(
    resumes[['resume_id', 'resume_text', 'hard_skills', 'profession', 'level']],
    on='resume_id',
    how='left',
    suffixes=('_vacancy', '_resume')
)

In [19]:
dataset = dataset.rename(columns={
    'profession_vacancy': 'vacancy_profession', 
    'profession_resume': 'resume_profession',    
    'level_vacancy': 'vacancy_level',            
    'level_resume': 'resume_level'              
})

In [20]:
dataset['vacancy_level'] = dataset['vacancy_level'].fillna('junior')
dataset['resume_level'] = dataset['resume_level'].fillna('junior')

dataset['required_skills_list'] = dataset['required_skills'].apply(parse_skills)
dataset['hard_skills_list'] = dataset['hard_skills'].apply(parse_skills)

In [21]:
print(test_df.columns.tolist())

NameError: name 'test_df' is not defined

In [22]:
print(dataset.columns.tolist())

['vacancy_id', 'resume_id', 'vacancy_profession', 'resume_profession', 'similarity', 'label', 'vacancy_salary', 'resume_salary', 'vacancy_text', 'required_skills', 'vacancy_profession', 'vacancy_level', 'resume_text', 'hard_skills', 'resume_profession', 'resume_level', 'required_skills_list', 'hard_skills_list']


In [23]:
# dataset['shared_count'] = dataset['shared_skills'].apply(len)
# dataset['skill_coverage'] = dataset.apply(
#     lambda row: row['shared_count'] / max(len(row['required_skills_list']), 1),
#     axis=1
# )
dataset['shared_skills'] = dataset.apply(
    lambda row: list(
        set(row['required_skills_list']) & set(row['hard_skills_list'])
    ),
    axis=1
)

dataset['shared_count'] = dataset['shared_skills'].apply(len)
dataset['skill_coverage'] = dataset.apply(
    lambda row: row['shared_count'] / max(len(row['required_skills_list']), 1),
    axis=1
)

In [24]:
dataset['label3'] = dataset['label'].map({0: 0, 1: 0, 2: 1, 3: 2})
print(dataset['label3'].value_counts().sort_index())


label3
0    323136
1     23420
2     13444
Name: count, dtype: int64


In [25]:
balanced_parts = []

for label in sorted(dataset['label3'].unique()):
    label_df = dataset[dataset['label3'] == label].copy()
    label_df = label_df.sample(frac=1, random_state=42)
    label_df = label_df.groupby('resume_id').head(MAX_PAIRS_PER_RESUME)
    label_df = label_df.groupby('vacancy_id').head(MAX_PAIRS_PER_VACANCY)
    balanced_parts.append(label_df)
    print(f"Класс {label}: {len(label_df)}")

dataset_balanced = pd.concat(balanced_parts, ignore_index=True)
dataset_balanced = dataset_balanced.loc[:, ~dataset_balanced.columns.duplicated()]

Класс 0: 18000
Класс 1: 5609
Класс 2: 10551


In [26]:
middle_df = dataset_balanced[dataset_balanced['label3'] == 1].copy()
augmented_rows = []

for _, row in middle_df.iterrows():
    row_dict = row.to_dict()
    augmented_rows.append(row_dict.copy())
    
    if random.random() < 0.5:
        new_row = row_dict.copy()
        skills = list(new_row.get('hard_skills_list', []))
        if len(skills) > 1:
            random.shuffle(skills)
            new_row['hard_skills_list'] = skills
        augmented_rows.append(new_row)

if augmented_rows:
    middle_aug_df = pd.DataFrame(augmented_rows)
    dataset_balanced = pd.concat([dataset_balanced, middle_aug_df], ignore_index=True)

In [27]:
target_size = dataset_balanced['label3'].value_counts().max()
equalized_parts = []

for label in sorted(dataset_balanced['label3'].unique()):
    class_df = dataset_balanced[dataset_balanced['label3'] == label].copy()
    
    if len(class_df) < target_size:
        extra = class_df.sample(target_size - len(class_df), replace=True, random_state=42)
        class_df = pd.concat([class_df, extra], ignore_index=True)
    elif len(class_df) > target_size:
        class_df = class_df.sample(target_size, random_state=42)
    
    equalized_parts.append(class_df)

dataset_balanced = pd.concat(equalized_parts, ignore_index=True)
dataset_balanced = dataset_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [28]:
train_df, temp_df = train_test_split(
    dataset_balanced, test_size=0.3, stratify=dataset_balanced['label3'], random_state=42
)
valid_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label3'], random_state=42
)

In [29]:
def build_texts(row):
    vacancy_skills = ", ".join(row['required_skills_list'][:10])
    resume_skills = ", ".join(row['hard_skills_list'][:10])
    shared_skills = ", ".join(row['shared_skills'][:8])

    vacancy_text = (
        f"Профессия: {row['vacancy_profession']}. "
        f"Уровень: {row['vacancy_level']}. "
        f"Требуемые навыки: {vacancy_skills}. "
        f"Описание: {str(row['vacancy_text'])[:400]}"
    )

    resume_text = (
        f"Профессия: {row['resume_profession']}. "
        f"Уровень: {row['resume_level']}. "
        f"Навыки: {resume_skills}. "
        f"Общие навыки: {shared_skills}. "
        f"Совпадение навыков: {row['shared_count']}. "
        f"Покрытие требований: {row['skill_coverage']:.2f}. "
        f"Опыт: {str(row['resume_text'])[:400]}"
    )

    return vacancy_text, resume_text

In [34]:
def prepare_samples(df):
    samples = []
    for _, row in df.iterrows():
        vacancy_text, resume_text = build_texts(row)
        samples.append(InputExample(
            texts=[vacancy_text, resume_text],
            label=int(row['label3'])
        ))
    return samples

train_samples = prepare_samples(train_df)
valid_samples = prepare_samples(valid_df)
test_samples = prepare_samples(test_df)

train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=BATCH_SIZE)

In [ ]:
# import sys
# !{sys.executable} -m pip install --upgrade "torch>=2.6" torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
   ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 GB 1.9 MB/s eta 0:21:59
   ---------------------------------------- 0.0/2.5 GB 1.5 MB/s eta 0:27:41
   ---------------------------------------- 0.0/2.5 GB 1.4 MB/s eta 0:29:49
   ---------------------------------------- 0.0/2.5 GB 1.7 MB/s eta 0:24:32
   ---------------------------------------- 0.0/2.5 GB 2.3 MB/s eta 0:18:30
   ---------------------------------------- 0.0/2.5 GB 3.0 MB/s eta 0:14:11
   ---------------------------------------- 0.0/2.5 GB 4.1 MB/s eta 0:10:25
   ---------------------------------------- 0.0/2.5 GB 5.1 MB/s eta 0:08:15
   ---------------------------------------- 0.0/2.5 GB 5.9 MB/s eta 0:07:12
   ---------------------------------------- 0.0/2.5 GB 5.9 MB/s eta 0:07:12
   ---------------------------------------- 0.0/2.5 GB 5.9 MB/s eta 0:07:12
   ---------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import torch
print(torch.__version__)

2.6.0+cu124


In [36]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = CrossEncoder(
    MODEL_NAME,
    num_labels=3,
    max_length=MAX_LENGTH,
    device=device,
    automodel_args={"ignore_mismatched_sizes": True}
)


The CrossEncoder `automodel_args` argument was renamed and is now deprecated. Please use `model_kwargs` instead.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22117.40it/s]


In [37]:
evaluator = None 


warmup_steps = math.ceil(len(train_dataloader) * EPOCHS * 0.1)

model.fit(
    train_dataloader=train_dataloader,
    evaluator=None,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path="hr_model_final",
    optimizer_params={'lr': LEARNING_RATE},
    use_amp=True if device == 'cuda' else False,
    show_progress_bar=True
)

Step,Training Loss
500,0.883770
1000,0.655168
1500,0.618356
2000,0.557027
2500,0.526490
3000,0.501955
3500,0.513283
4000,0.490279
4500,0.468656
5000,0.462458


In [38]:

test_pairs = [(s.texts[0], s.texts[1]) for s in test_samples]
predictions = model.predict(test_pairs)
pred_labels = np.argmax(predictions, axis=1)
true_labels = [int(s.label) for s in test_samples]

macro_f1 = f1_score(true_labels, pred_labels, average='macro')
weighted_f1 = f1_score(true_labels, pred_labels, average='weighted')
accuracy = accuracy_score(true_labels, pred_labels)


print(f"\Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")

model.save_pretrained("model3")
test_examples = [
    ("Python разработчик опыт 3 года Django", "Пишу на Python 5 лет, работал с Django и Flask"),
    ("Senior Java разработчик", "Умею писать на Python, но Java не знаю"),
    ("Требуется водитель категории B", "Опыт вождения 10 лет, все права есть"),
]

for vac, res in test_examples:
    pred = model.predict([(vac, res)])[0]
    class_id = np.argmax(pred)
    class_names = ["Не подходит", "Возможен", "Подходит"]
    print(f"\nВакансия: {vac}")
    print(f"Резюме: {res}")
    print(f"Результат: {class_names[class_id]} (уверенность: {pred[class_id]:.3f})")


\Macro F1: 0.9087
Weighted F1: 0.9087
Accuracy: 0.9078


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]


Вакансия: Python разработчик опыт 3 года Django
Резюме: Пишу на Python 5 лет, работал с Django и Flask
Результат: Подходит (уверенность: 4.559)

Вакансия: Senior Java разработчик
Резюме: Умею писать на Python, но Java не знаю
Результат: Возможен (уверенность: 1.965)

Вакансия: Требуется водитель категории B
Резюме: Опыт вождения 10 лет, все права есть
Результат: Возможен (уверенность: 2.275)
